In [ ]:
# The SQL query to collect production backtests from the database can be found in the client issues log

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/backtests/prod_hist_pvnet_backtest_2024-09_to_2026-02/data-1771432727184.csv")
df

In [ ]:
# Find top generation to know which days to remove

pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', None)

def get_top_n_by_generation_mw(df, n=50):
    """
    Returns the top n rows of the dataframe ordered by generation_mw (descending).
    """
    return df.sort_values(by="generation_mw", ascending=False).head(n)

top_10_largest = get_top_n_by_generation_mw(df, n=50)

print(top_10_largest["forecast_creation_datetime_utc"].unique())

top_10_largest

In [ ]:
# We want a function with mimics the handling of the forecast via the API, correct as of 18/02/2026
# 1. Limit the adjuster values by + and - 1000
# 2. Don't allow generation values less than 0
# 3. Also round to two decimal places
def adjust_and_clean_forecasts_df(df):
    """
    Adjust the forecast dataframe according to the API handling rules as of 18/02/2026:
    1. Limit the adjuster values by + and - 1000
    2. Don't allow generation values less than 0
    3. Round to two decimal places
    4. Remove period where pvlive had capacity issues
    """
    df = df.copy()

    df["adjust_mw"] = df["adjust_mw"].clip(lower=-1000, upper=1000)

    df["generation_mw"] = df["generation_mw"] - df["adjust_mw"]

    df["generation_mw"] = df["generation_mw"].clip(lower=0).round(2)

    df["generation_mw_p10"] = df["generation_mw_p10"].round(2)
    df["generation_mw_p90"] = df["generation_mw_p90"].round(2)
    
    df = df.drop(columns=["adjust_mw"])

    # List of forecast_creation_times with PVLive capacity issues
    pvlive_issues_times = ['2025-02-03 19:03:54.024044+00','2025-02-03 18:33:36.84963+00', '2025-02-03 19:48:05.451964+00', '2025-02-03 20:23:41.291614+00']

    df = df[~df["forecast_creation_datetime_utc"].isin(pvlive_issues_times)]

    df = df.sort_values(by=["forecast_creation_datetime_utc", "start_datetime_utc"])

    return df



In [ ]:
df_proc = adjust_and_clean_forecasts_df(df)
df_proc

In [ ]:
# Check if 'forecast_creation_datetime_utc' is in order
is_ordered = df_proc["forecast_creation_datetime_utc"].is_monotonic_increasing
is_ordered

In [ ]:
# Save file as a csv/parquet file, complete and then also yearly if needed.
# Can zip files afterwards also
path = "../data/backtests/prod_hist_pvnet_backtest_2024-09_to_2024-02/"

df_proc.to_csv(path + "quartz_solar_production_backtest_uk_national_model=blend+adj_start=2024_09_01_end=2025_02_18_created=2025_02_29.csv", index=False)
df_proc.to_parquet(path + "quartz_solar_production_backtest_uk_national_model=blend+adj_start=2024_09_01_end=2025_02_18_created=2025_02_29.parquet", engine='fastparquet', index=False)